# 02 — Exploration du dataset IBM

**Objectif :** comprendre le dataset avant de l'utiliser : période couverte, taux de fraude,
canaux de paiement, achats en ligne et à l'étranger, montants et erreurs.
Ces réponses décideront du découpage train / validation / test et du géocodage.

**Source :** couche bronze `data/processed/bronze/transactions.parquet` (24 386 900 transactions).**

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)

# Diagnostic : ou tourne ce notebook, et avec quel Python ?
print("Python utilise    :", sys.executable)
print("Dossier courant   :", Path.cwd())

# Racine du projet : on cherche .git au-dessus du dossier courant,
# sinon on utilise le chemin du projet affiche par le terminal Ubuntu.
PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)

BRONZE = ROOT / "data" / "processed" / "bronze" / "transactions.parquet"
FIGURES = ROOT / "docs" / "figures"

print("Racine du projet  :", ROOT)
print("Fichier bronze OK :", BRONZE.exists())

In [ ]:
if not BRONZE.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {BRONZE}\n"
        "-> Verifie que le noyau est 'secumon (Python 3.14)' et que l'etape 1.2 "
        "(conversion en Parquet) a bien ete faite."
    )

FIGURES.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE VIEW tx AS
    SELECT
        CAST("User" AS INTEGER)                      AS user_id,
        CAST("Card" AS INTEGER)                      AS card_idx,
        CAST("Year" AS INTEGER)                      AS year,
        CAST(split_part("Time", ':', 1) AS INTEGER)  AS hour,
        CAST(REPLACE("Amount", '$', '') AS DOUBLE)   AS amount,
        "Use Chip"                                   AS use_chip,
        "Merchant City"                              AS merchant_city,
        "Merchant State"                             AS merchant_state,
        "MCC"                                        AS mcc,
        "Errors?"                                    AS errors,
        ("Is Fraud?" = 'Yes')                        AS is_fraud
    FROM '{BRONZE}'
""")
print("Vue 'tx' creee.")
con.sql("SELECT * FROM tx LIMIT 5").df()

## 1. Vue d'ensemble

In [ ]:
con.sql("""
    SELECT COUNT(*)                                                  AS transactions,
           COUNT(DISTINCT user_id)                                   AS clients,
           COUNT(DISTINCT CAST(user_id AS VARCHAR) || '-' || CAST(card_idx AS VARCHAR)) AS cartes,
           MIN(year)                                                 AS annee_min,
           MAX(year)                                                 AS annee_max,
           SUM(is_fraud::INT)                                        AS fraudes,
           ROUND(100 * AVG(is_fraud::INT), 3)                        AS taux_fraude_pct
    FROM tx
""").df()

## 2. Fraude par année
Sert à choisir les dates de coupure train / validation / test : chaque période
doit contenir assez de fraudes pour entraîner et évaluer les modèles.

In [ ]:
by_year = con.sql("""
    SELECT year,
           COUNT(*)                            AS transactions,
           SUM(is_fraud::INT)                  AS fraudes,
           ROUND(100 * AVG(is_fraud::INT), 3)  AS taux_fraude_pct
    FROM tx
    GROUP BY year
    ORDER BY year
""").df()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(by_year["year"], by_year["fraudes"])
ax.set_xlabel("Année")
ax.set_ylabel("Nombre de fraudes")
ax.set_title("Fraudes par année — dataset IBM")
fig.tight_layout()
fig.savefig(FIGURES / "eda_fraudes_par_annee.png", dpi=150)
plt.show()

by_year

## 3. Canal de paiement
`Use Chip` indique comment la carte a été utilisée : piste magnétique (Swipe), puce (Chip)
ou en ligne (Online). On compare la part de chaque canal et son taux de fraude.

In [ ]:
con.sql("""
    SELECT use_chip                                          AS canal,
           COUNT(*)                                          AS transactions,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS part_pct,
           ROUND(100 * AVG(is_fraud::INT), 3)                AS taux_fraude_pct
    FROM tx
    GROUP BY use_chip
    ORDER BY transactions DESC
""").df()

## 4. Achats en ligne et commerçants à l'étranger
Hypothèse à vérifier : `Merchant State` contient un code à 2 lettres pour les États-Unis,
un nom de pays pour l'étranger, et reste vide pour les achats en ligne.

In [ ]:
zones = con.sql("""
    SELECT CASE
             WHEN merchant_state IS NULL OR merchant_state = '' THEN 'vide (en ligne ?)'
             WHEN length(merchant_state) = 2                    THEN 'Etats-Unis (code a 2 lettres)'
             ELSE 'etranger (nom de pays)'
           END                                               AS zone,
           COUNT(*)                                          AS transactions,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS part_pct,
           ROUND(100 * AVG(is_fraud::INT), 3)                AS taux_fraude_pct
    FROM tx
    GROUP BY zone
    ORDER BY transactions DESC
""").df()

vides = con.sql("""
    SELECT merchant_city, use_chip AS canal, COUNT(*) AS transactions
    FROM tx
    WHERE merchant_state IS NULL OR merchant_state = ''
    GROUP BY merchant_city, use_chip
    ORDER BY transactions DESC
    LIMIT 5
""").df()

pays = con.sql("""
    SELECT merchant_state                       AS pays,
           COUNT(*)                             AS transactions,
           ROUND(100 * AVG(is_fraud::INT), 2)   AS taux_fraude_pct
    FROM tx
    WHERE length(merchant_state) > 2
    GROUP BY merchant_state
    ORDER BY transactions DESC
    LIMIT 15
""").df()

print("Repartition par zone :")
display(zones)
print("Contenu des lignes sans 'Merchant State' :")
display(vides)
print("Pays etrangers les plus frequents :")
display(pays)

## 5. Montants
On regarde l'étendue des montants, la médiane (moins sensible aux valeurs extrêmes que la moyenne),
la présence de montants négatifs (remboursements ?) et le montant moyen des fraudes.

In [ ]:
con.sql("""
    SELECT ROUND(MIN(amount), 2)                                AS montant_min,
           ROUND(MAX(amount), 2)                                AS montant_max,
           ROUND(AVG(amount), 2)                                AS moyenne,
           ROUND(MEDIAN(amount), 2)                             AS mediane,
           SUM((amount < 0)::INT)                               AS montants_negatifs,
           ROUND(AVG(CASE WHEN is_fraud THEN amount END), 2)    AS moyenne_fraudes,
           ROUND(AVG(CASE WHEN NOT is_fraud THEN amount END), 2) AS moyenne_legitimes
    FROM tx
""").df()

## 6. Erreurs de transaction
Les erreurs (mauvais PIN, solde insuffisant...) serviront à calculer des features
comme le nombre d'échecs récents d'une carte.

In [ ]:
con.sql("""
    SELECT COALESCE(errors, '(aucune)')        AS erreur,
           COUNT(*)                            AS transactions,
           ROUND(100 * AVG(is_fraud::INT), 2)  AS taux_fraude_pct
    FROM tx
    GROUP BY erreur
    ORDER BY transactions DESC
    LIMIT 10
""").df()

## 7. Catégories de commerçants (MCC) les plus touchées
Seules les catégories avec au moins 10 000 transactions sont gardées,
pour éviter les taux trompeurs calculés sur trop peu de cas.

In [ ]:
con.sql("""
    SELECT mcc,
           COUNT(*)                            AS transactions,
           SUM(is_fraud::INT)                  AS fraudes,
           ROUND(100 * AVG(is_fraud::INT), 3)  AS taux_fraude_pct
    FROM tx
    GROUP BY mcc
    HAVING COUNT(*) >= 10000
    ORDER BY taux_fraude_pct DESC
    LIMIT 10
""").df()

## 8. Fraude selon l'heure de la journée

In [ ]:
by_hour = con.sql("""
    SELECT hour,
           COUNT(*)                            AS transactions,
           ROUND(100 * AVG(is_fraud::INT), 3)  AS taux_fraude_pct
    FROM tx
    GROUP BY hour
    ORDER BY hour
""").df()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(by_hour["hour"], by_hour["taux_fraude_pct"], marker="o")
ax.set_xticks(range(0, 24))
ax.set_xlabel("Heure")
ax.set_ylabel("Taux de fraude (%)")
ax.set_title("Taux de fraude selon l'heure — dataset IBM")
fig.tight_layout()
fig.savefig(FIGURES / "eda_fraude_par_heure.png", dpi=150)
plt.show()

by_hour

## 9. Vérifications avant le découpage
Combien de fraudes dans chaque période ? Les fraudes sont-elles concentrées sur quelques cartes ?

In [ ]:
SPLIT_SQL = """
    CASE WHEN year <= 2014 THEN '1_train (1991-2014)'
         WHEN year =  2015 THEN '2_validation (2015)'
         WHEN year =  2016 THEN '3_test_A connu (2016)'
         ELSE                   '4_test_B campagne (2017-2020)'
    END
"""

split = con.sql(f"""
    SELECT {SPLIT_SQL}                           AS periode,
           COUNT(*)                              AS transactions,
           SUM(is_fraud::INT)::BIGINT            AS fraudes,
           ROUND(100 * AVG(is_fraud::INT), 3)    AS taux_fraude_pct,
           COUNT(DISTINCT user_id)               AS clients
    FROM tx
    GROUP BY periode
    ORDER BY periode
""").df()
split

In [ ]:
cartes = con.sql("""
    WITH c AS (
        SELECT user_id, card_idx, SUM(is_fraud::INT)::BIGINT AS f
        FROM tx
        GROUP BY user_id, card_idx
    )
    SELECT COUNT(*)                                     AS cartes,
           SUM((f > 0)::INT)                            AS cartes_fraudees,
           ROUND(AVG(CASE WHEN f > 0 THEN f END), 1)    AS fraudes_moy_par_carte_fraudee,
           MAX(f)                                       AS max_fraudes_une_carte
    FROM c
""").df()

italie = con.sql("""
    SELECT COUNT(DISTINCT user_id)                                                     AS clients,
           COUNT(DISTINCT CAST(user_id AS VARCHAR) || '-' || CAST(card_idx AS VARCHAR)) AS cartes,
           SUM(is_fraud::INT)::BIGINT                                                  AS fraudes,
           MIN(year)                                                                   AS premiere_annee,
           MAX(year)                                                                   AS derniere_annee
    FROM tx
    WHERE merchant_state = 'Italy' AND is_fraud
""").df()

print("Concentration des fraudes par carte :")
display(cartes)
print("Fraudes en Italie : combien de clients et de cartes ?")
display(italie)

In [ ]:
con.sql("""
    SELECT CASE WHEN year <= 2015 THEN '1_train'
                WHEN year <= 2017 THEN '2_validation'
                ELSE                   '3_test'
           END                                                     AS periode,
           CASE WHEN merchant_state = 'Italy'                      THEN 'Italie'
                WHEN merchant_state IS NULL OR merchant_state = '' THEN 'en ligne'
                WHEN length(merchant_state) = 2                    THEN 'Etats-Unis'
                ELSE 'autre etranger'
           END                                                     AS zone,
           SUM(is_fraud::INT)::BIGINT                              AS fraudes
    FROM tx
    GROUP BY periode, zone
    ORDER BY periode, fraudes DESC
""").df().pivot(index="periode", columns="zone", values="fraudes").fillna(0).astype(int)

## 10. Conclusions de l'exploration

**Données** : 24 386 900 transactions, 2 000 clients, 6 139 cartes, 1991 – début 2020.
29 757 fraudes, soit 0,122 % → classes très déséquilibrées : évaluation par précision, rappel et PR-AUC, jamais par accuracy.

**Découpage temporel retenu**

| Jeu | Années | Transactions | Fraudes | Usage |
|---|---|---|---|---|
| Train | 1991–2014 | ? | 18 064 | apprentissage |
| Validation | 2015 | ? | 3 281 | hyperparamètres et seuils |
| Test A (connu) | 2016 | ? | 3 579 | performance sur des fraudes de type connu |
| Test B (campagne) | 2017–2020 | ? | 4 833 | détection d'une campagne jamais vue (Italie) |

**Pourquoi deux jeux de test** : la fraude en ligne et aux États-Unis s'arrête en 2016 ;
de 2017 à 2019, 96 % des fraudes viennent d'une campagne en Italie (432 clients, 496 cartes), absente du train.
Un seul chiffre global ne mesurerait qu'un scénario.

**Observations**
- Canal : en ligne 0,676 % de fraude, puce 0,077 %, piste magnétique 0,043 %.
- Zone : États-Unis 0,023 %, en ligne 0,674 %, étranger 4,134 %.
- Fraude en rafales : 45 % des cartes touchées au moins une fois, 10,8 fraudes en moyenne par carte fraudée → variables de vélocité.
- Montant moyen : fraudes 108,59 contre 43,55 pour les légitimes → montant **comparé aux habitudes du client**, pas un seuil fixe.
- Erreurs CVV (2,61 %), date d'expiration (1,12 %), numéro de carte (0,79 %), PIN (0,51 %) → card testing → nombre d'erreurs récentes par carte.
- MCC les plus touchés : électronique (5732) et voyage (locations de voitures, hôtels).
- Heure : effet modéré → par rapport aux habitudes du client.
- Pas de pays brut dans le modèle : variables contextuelles (pays inhabituel, distance, voyage impossible).

**À nettoyer dans la couche silver**
- Montant : `$` → nombre ; 1 244 683 montants négatifs → indicateur `is_refund`, exclus des profils de dépense.
- 7 601 transactions `Chip Transaction` avec la ville `ONLINE` (incohérent).
- Reconstruire un horodatage complet (année, mois, jour, heure, minute).
- `Zip` au format `91750.0` → texte à 5 chiffres.
- Jointure avec les tables clients et cartes, en supprimant les données personnelles (nom, adresse, CVV) et en tokenisant le numéro de carte.

In [ ]:
con.sql("""
    SELECT year,
           CASE WHEN merchant_state = 'Italy'                      THEN 'Italie'
                WHEN merchant_state IS NULL OR merchant_state = '' THEN 'en ligne'
                WHEN length(merchant_state) = 2                    THEN 'Etats-Unis'
                ELSE 'autre etranger'
           END                          AS zone,
           SUM(is_fraud::INT)::BIGINT   AS fraudes
    FROM tx
    WHERE year >= 2012
    GROUP BY year, zone
""").df().pivot(index="year", columns="zone", values="fraudes").fillna(0).astype(int)


- Clients : 1 538 dans le train, mais 1 959 dans le test B → environ 400 clients apparaissent
  seulement après 2014 (démarrage à froid). Les profils clients seront donc calculés uniquement
  avec l'historique disponible **avant** chaque transaction, jamais avec un profil figé sur le train.